# 01 – Delta Update

Inkrementelles Update der Delta Tables aus einem neuen Polar-Export-ZIP.

**Regelmässig ausführen** – z.B. nach jedem neuen Datenexport von Polar.

## Ablauf
1. Secrets & Verbindung prüfen
2. ZIP-Dateien in `input/` erkennen
3. MD5-Hashes gegen `import_log` prüfen (nur neue/geänderte Dateien)
4. DataFrames per MERGE INTO in Delta Tables schreiben
5. Import-Log aktualisieren
6. ZIP nach `archive/` verschieben
7. Delta-Bericht & Tabellenstand ausgeben

---
> 📦 **Vorbereitung:** Neues Polar-Export-ZIP in den `input/` Ordner legen,  
> dann alle Zellen von oben nach unten ausführen (`Run All`).

## 1 – Imports & Secrets prüfen

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

# src/ zum Python-Pfad hinzufügen
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from db_loader import DatabricksLoader, secrets_pruefen
from delta_updater import DeltaUpdater

# ── Secrets prüfen ──────────────────────────────────────────
secrets_pruefen()

print(f"\nNotebook gestartet: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2 – ZIP-Dateien erkennen

In [ ]:
input_ordner = Path('..') / 'input'
zip_dateien  = sorted(input_ordner.glob('*.zip'))

print(f"📂 Suche ZIP-Dateien in: {input_ordner.resolve()}")
print()

if not zip_dateien:
    print("⚠️  Keine ZIP-Dateien gefunden!")
    print()
    print("So exportierst du deine Polar-Daten:")
    print("  1. Polar Flow Web (flow.polar.com) oder Polar App öffnen")
    print("  2. Profil → Einstellungen → Eigene Daten exportieren")
    print("  3. ZIP herunterladen")
    print("  4. Datei in den 'input/' Ordner dieses Projekts kopieren")
    print("  5. Diese Zelle erneut ausführen")
else:
    print(f"Gefunden: {len(zip_dateien)} ZIP-Datei(en)")
    print()
    for z in zip_dateien:
        groesse_mb = z.stat().st_size / 1024 / 1024
        mtime      = datetime.fromtimestamp(z.stat().st_mtime)
        print(f"  📦 {z.name}")
        print(f"     Grösse   : {groesse_mb:.1f} MB")
        print(f"     Geändert : {mtime.strftime('%Y-%m-%d %H:%M')}")

## 3 – Vorschau: Was ist neu?

Vergleicht die ZIP-Inhalte mit dem Import-Log, **ohne** Daten zu schreiben.

In [ ]:
import hashlib
import zipfile
import re

if not zip_dateien:
    print("⏭️  Kein ZIP vorhanden – Zelle übersprungen")
else:
    # Import-Log laden
    db = DatabricksLoader()
    df_log = db.abfrage(
        f"SELECT dateiname, hash_md5, kategorie "
        f"FROM {db.catalog}.{db.schema}.import_log"
    )
    bekannte_hashes = dict(zip(df_log['dateiname'], df_log['hash_md5'])) \
                      if not df_log.empty else {}
    print(f"Import-Log: {len(bekannte_hashes)} bekannte Dateien")
    print()

    # Vorschau pro ZIP
    for zip_pfad in zip_dateien:
        print(f"── {zip_pfad.name} ──")
        zaehler = {'neu': 0, 'geaendert': 0, 'unveraendert': 0}
        kategorien_neu = {}

        with zipfile.ZipFile(zip_pfad, 'r') as zf:
            json_dateien = [n for n in zf.namelist() if n.endswith('.json')]

            for name in json_dateien:
                with zf.open(name) as f:
                    h = hashlib.md5(f.read()).hexdigest()

                if name not in bekannte_hashes:
                    zaehler['neu'] += 1
                    # Kategorie bestimmen
                    if   'activity_' in name: kat = 'activity'
                    elif 'training_' in name: kat = 'training'
                    elif '247ohr_'   in name: kat = 'heartrate'
                    elif 'ppi_'      in name: kat = 'hrv'
                    else:                     kat = 'sonstige'
                    kategorien_neu[kat] = kategorien_neu.get(kat, 0) + 1
                elif bekannte_hashes[name] != h:
                    zaehler['geaendert'] += 1
                else:
                    zaehler['unveraendert'] += 1

        print(f"  JSON-Dateien gesamt : {len(json_dateien)}")
        print(f"  Neu                 : {zaehler['neu']}")
        print(f"  Geändert            : {zaehler['geaendert']}")
        print(f"  Unverändert         : {zaehler['unveraendert']}")
        if kategorien_neu:
            print(f"  Neue Dateien nach Kategorie:")
            for kat, n in sorted(kategorien_neu.items()):
                print(f"    {kat:<12}: {n}")
        else:
            print("  ℹ️  Keine neuen Dateien – ZIP bereits vollständig importiert")
        print()

## 4 – Delta Update ausführen

> ✅ Nur ausführen wenn Schritt 3 neue/geänderte Dateien angezeigt hat.

In [ ]:
if not zip_dateien:
    print("⏭️  Kein ZIP vorhanden – Zelle übersprungen")
else:
    updater  = DeltaUpdater()
    berichte = []

    for zip_pfad in zip_dateien:
        bericht = updater.verarbeite_zip(str(zip_pfad))
        berichte.append({'datei': zip_pfad.name, **bericht})

## 5 – Zusammenfassung

In [ ]:
import pandas as pd

if not zip_dateien:
    print("⏭️  Kein ZIP verarbeitet")
else:
    # Bericht-Tabelle
    df_bericht = pd.DataFrame(berichte)
    print("📊 Import-Bericht:")
    display(df_bericht)

    # Gesamtzahlen
    gesamt_neu      = df_bericht['neu'].sum()
    gesamt_geaendert = df_bericht['geaendert'].sum()
    gesamt_dauer    = df_bericht['dauer_sekunden'].sum()
    print(f"\nGesamt neu importiert : {gesamt_neu}")
    print(f"Gesamt geändert       : {gesamt_geaendert}")
    print(f"Gesamtdauer           : {gesamt_dauer:.1f}s")

## 6 – Tabellenstand nach Update

In [ ]:
# Aktuellen Stand aller Tabellen anzeigen
db2 = DatabricksLoader()
df_stand = db2.tabellen_uebersicht()
display(df_stand)

# Import-Log Übersicht
print("\n── Import-Log nach Update ──")
display(db2.import_log_uebersicht())

# Archiv prüfen
archiv_ordner = Path('..') / 'archive'
archiv_zips   = list(archiv_ordner.glob('*.zip'))
print(f"\n📁 Archiv: {len(archiv_zips)} ZIP-Datei(en)")
for z in sorted(archiv_zips):
    groesse_mb = z.stat().st_size / 1024 / 1024
    print(f"   {z.name}  ({groesse_mb:.1f} MB)")

db2.schliessen()
print("\n✅ Delta Update abgeschlossen – weiter mit 02_exploration.ipynb")